# Prepare BARRA-C2 data for analysis

- Weight data by population for each state

In [1]:
import xarray as xr
import os

### Load population and region mask

In [2]:
nem_pop = xr.open_dataset(
    "/home/565/ad1803/Hot_Cloudy/Demand/input_data/NEM_population_density_state_mask_barra-c2_grid.nc"
)["population_density"]

Prepare masks

In [3]:
def scale(da):
    """
    Scale da to be between 0 and 1. AD: We scale data so we can compare datasets with different units or apply machiene learning
    """
    return (da - da.min(dim=["lat", "lon"])) / (da.max(dim=["lat", "lon"]) - da.min(dim=["lat", "lon"])) #AD: standard scaling formula

    #AD: da.min(dim=["lat", "lon"])---This calculates the minimum value of da across the latitude and longitude dimensions. So if da is a 3D array with dimensions like [time, lat, lon], you're collapsing over the spatial dimensions and getting a 1D array over time, each with the spatial min at that timestep.

In [4]:
def prepare_mask(mask):
    """
    Scale mask to be 0-1, set NaNs to zero and chunk.
    """
    mask = mask.where(mask.notnull(), 0) #AD: replaces NaN values with 0 
    return scale(mask).chunk() #AD: this returns using the scale function defined above, the mask defined in the line above. chunk() converts xarray DataArray to Dask array, used for large datasets on NCI

In [5]:
nem_pop = prepare_mask(nem_pop) #AD: This redfines nem_pop to be the data set after scaling and masking

### Weight predictors and average over regions

In [6]:
def open_hourly(fp, lat_slice, lon_slice, lat_name="lat", lon_name="lon"):
    """
    Open multiple hourly files and preprocess to region.
    
    fp: str, path to file. Should not include files, only the path to dir.
    lat_slice, lon_slice: slice of lat/lon to subset
    lat_name, lon_name: names of lat/lon coords.
    """
    def preprocess(ds):
        ds = ds.rename({lat_name: "lat"}) 
        ds = ds.rename({lon_name: "lon"})
        return ds.sel(lon=lon_slice, lat=lat_slice)  #AD: this renames lat and lon data to exact names as will be referenced throughout the code
    
    ds = xr.open_mfdataset(
        fp,
        preprocess=preprocess,
        chunks={"time": "200MB"} #AD: this is relling Dask to chunk along the time dimension to best handle memory 
    )
    return ds

In [7]:
def population_mean(ds, mask):
    """
    Compute the spatial average of population-weighted data for each region.
    
    ds: dataset or array to process
    mask: population mask/data
    """
    return ds.weighted(mask).mean(["lat", "lon"]) #AD: .weighted(mask)---Applies a spatial weight to your data using the values in mask    .mean(["lat", "lon"]---Takes the mean across the spatial dimensions — latitude and longitude

In [8]:
def write(ds, fp):
    """
    Write to file and chunk to single chunk.
    
    ds: dataset to write.
    fp: str, path to write to.
    """
    ds = ds.chunk({"time": -1}) #AD: the -1 means to load it all in the one block on the time dimension
    ds.to_netcdf(fp + ".nc")

Dictionary of variables to process.

Organised with the variable name as the key, then a list as follows:
`[path_to_open, lat_name, lon_name, path_to_write, var_name, new_var_name]`

In [9]:
years = range(1994, 2024)

In [12]:
barra_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/"

In [13]:
write_path = "/home/565/ad1803/dunkelflaute/casestudy/data/hurs/"

In [14]:
fy = str(min(years))  # "2009"
ly = str(max(years))  # "2023"

variables = {
    "hurs": [
        barra_path + "hurs/latest/",  # or with "*.nc" as needed
        "lat",
        "lon",
        write_path + "hurs/hurs_barra-c2_hourly_" + fy + "-" + ly + "_NEM",
        "hurs",
        "hurs"
    ]
}

Loop through and process

In [15]:
month_str = [ "04", "05", "06", "07", "08", "09"]

In [17]:
for key, values in zip(variables.keys(), variables.values()):
    
    for year in years:
        files_to_open = [
    f"{values[0]}hurs_AUST-04_ERA5_historical_hres_BOM_BARRA-C2_v1_1hr_{year}{m}-{year}{m}.nc"
    for m in month_str
]
    
        ds = open_hourly(files_to_open, slice(-44, -10), slice(125, 154), values[1], values[2])
        ds = population_mean(ds, nem_pop)
        ds = ds.rename({values[4]: values[5]})

        fp = values[3] + "_" + "pop_dens_mask_" + str(year)
        print(year)
        write(ds, fp)

1994


PermissionError: [Errno 13] Permission denied: '/home/565/ad1803/dunkelflaute/casestudy/data/hurs/hurs/hurs_barra-c2_hourly_1994-2023_NEM_pop_dens_mask_1994.nc'

# Close cluster